# DriftWatch: UCI Bank Marketing EDA, Cleaning, Training, and MLflow

This notebook is the end-to-end model development record for the platform service.

- Dataset: `platform/data/bank-additional-full.csv` only.
- Requirement source: `driftwatch requirements.pdf`.
- Split rule: stratified 60/20/20 with `random_state=42`.
- Target: `y`, where `yes` means the client subscribed to a term deposit.
- Operating rule: choose the highest threshold that still satisfies `recall >= 0.75`.


## 1. Setup

The notebook keeps EDA helper functions local so the exploratory section is readable and self-contained. Production cleaning, training, tuning, thresholding, artifact writing, and MLflow registration come from `platform/app/ml` so the notebook and service use the same implementation.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "platform").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PLATFORM_ROOT = PROJECT_ROOT / "platform"
SOURCE_DIR = PROJECT_ROOT / "docs - DO NOT COMMIT" / "Case-Study-UCI-Bank-Marketing-Dataset"

sys.path.insert(0, str(PLATFORM_ROOT))

from app.config import get_ml_settings
from app.ml.data import (
    clean_bank_marketing_data,
    load_bank_marketing_data,
    make_train_validation_test_split,
    split_features_target,
)
from app.ml.train import run_training_pipeline

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

DATA_PATH = PLATFORM_ROOT / "data" / "bank-additional-full.csv"
NAMES_PATH = PLATFORM_ROOT / "data" / "bank-additional-names.txt"
ARTIFACT_ROOT = PLATFORM_ROOT / "artifacts"


## 2. Dataset Context

The UCI notes describe this as phone-call campaign data from a Portuguese bank. The social and economic fields are important for DriftWatch because they can move over time and create realistic drift signals.

Key project constraints:

- Use `bank-additional-full.csv`, not the older `bank-full.csv` schema.
- Do not use `bank-additional.csv` as a test set because it is a random 10 percent sample from the full file, so it is not independent.
- Drop `duration`; it is known only after a call ends and leaks the label.
- Treat `pdays == 999` as a sentinel meaning the client was not previously contacted.
- Keep `unknown` as a real categorical value because the requirements and UCI notes say it is informative.


In [ ]:
print(DATA_PATH)
print(NAMES_PATH)

names_text = NAMES_PATH.read_text(encoding="utf-8-sig")
print("bank-additional-full.csv" in names_text)
print("Number of characters in data notes:", len(names_text))


## 3. Load Raw Data

We inspect the raw file before cleaning so the notebook can prove why each cleaning decision exists.


In [ ]:
raw_df = load_bank_marketing_data(DATA_PATH)

display(raw_df.shape)
display(raw_df.head())
display(raw_df.tail())


## 4. Notebook EDA Helpers

These helpers are intentionally scoped to the notebook. They produce compact tables and simple plots that are useful for model decisions without turning the EDA into a long plotting exercise.


In [ ]:
TARGET_COL = "y"
POSITIVE_LABEL = "yes"
NEGATIVE_LABEL = "no"


def feature_types(df, target_col=TARGET_COL):
    features = df.drop(columns=[target_col], errors="ignore")
    numeric_cols = features.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = [col for col in features.columns if col not in numeric_cols]
    return numeric_cols, categorical_cols


def column_overview(df):
    return pd.DataFrame(
        {
            "column": df.columns,
            "dtype": [str(dtype) for dtype in df.dtypes],
            "missing_count": df.isna().sum().to_numpy(),
            "missing_percent": (df.isna().sum() / len(df) * 100).round(2).to_numpy(),
            "unique_count": df.nunique(dropna=False).to_numpy(),
        }
    )


def unknown_summary(df):
    rows = []
    for column in df.select_dtypes(include="object").columns:
        count = int((df[column] == "unknown").sum())
        if count:
            rows.append(
                {
                    "column": column,
                    "unknown_count": count,
                    "unknown_percent": round(count / len(df) * 100, 2),
                    "unique_count": int(df[column].nunique(dropna=False)),
                }
            )
    return pd.DataFrame(rows).sort_values("unknown_percent", ascending=False)


def target_rate_table(df, column, target_col=TARGET_COL, min_count=25):
    work = df[[column, target_col]].copy()
    work["target_yes"] = work[target_col].map({POSITIVE_LABEL: 1, NEGATIVE_LABEL: 0})
    out = (
        work.groupby(column, dropna=False)
        .agg(count=("target_yes", "size"), yes_rate=("target_yes", "mean"))
        .query("count >= @min_count")
        .sort_values("yes_rate", ascending=False)
        .reset_index()
    )
    out["yes_rate"] = (out["yes_rate"] * 100).round(2)
    return out


def numeric_target_correlation(df, target_col=TARGET_COL):
    work = df.copy()
    work["target_yes"] = work[target_col].map({POSITIVE_LABEL: 1, NEGATIVE_LABEL: 0})
    corr = work.select_dtypes(include="number").corr(numeric_only=True)["target_yes"]
    return corr.drop("target_yes").sort_values(key=lambda s: s.abs(), ascending=False).round(3)


def plot_target_rate(df, column, min_count=25, top_n=12):
    table = target_rate_table(df, column, min_count=min_count).head(top_n)
    plt.figure(figsize=(9, 4))
    sns.barplot(data=table, x="yes_rate", y=column, color="#4C78A8")
    plt.title(f"Subscription rate by {column}")
    plt.xlabel("Yes rate (%)")
    plt.ylabel(column)
    plt.tight_layout()
    plt.show()
    display(table)


## 5. Basic Structure

The full dataset has 41,188 rows and 20 input features plus the target. That matches the requirements PDF and confirms we are using the correct UCI file.


In [ ]:
numeric_cols, categorical_cols = feature_types(raw_df)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]:,}")
print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

display(column_overview(raw_df))


## 6. Target Distribution

The positive class is intentionally rare, around 11 percent. Accuracy alone would be misleading, so the project uses a recall-constrained threshold.


In [ ]:
target_counts = raw_df[TARGET_COL].value_counts().rename_axis("target").reset_index(name="count")
target_counts["percent"] = (target_counts["count"] / len(raw_df) * 100).round(2)
display(target_counts)

plt.figure(figsize=(6, 4))
sns.barplot(data=target_counts, x="target", y="count", hue="target", palette=["#8E8E8E", "#4C78A8"], legend=False)
plt.title("Target class balance")
plt.xlabel("Subscribed term deposit")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()


## 7. Data Quality Checks

The UCI file uses the string `unknown` rather than nulls for several categorical fields. We keep those values as categories instead of imputing them away.


In [ ]:
quality_summary = pd.DataFrame(
    {
        "row_count": [len(raw_df)],
        "duplicate_rows": [int(raw_df.duplicated().sum())],
        "missing_cells": [int(raw_df.isna().sum().sum())],
    }
)
display(quality_summary)
display(unknown_summary(raw_df))


## 8. Leakage and Sentinel Analysis

Two columns need explicit handling before modeling:

- `duration` is highly predictive because it is measured after the call. It must be dropped for a realistic serving model.
- `pdays == 999` means no previous campaign contact. We keep a flag for that meaning and replace the sentinel in the numeric cleaned copy.


In [ ]:
work = raw_df.copy()
work["target_yes"] = work[TARGET_COL].map({POSITIVE_LABEL: 1, NEGATIVE_LABEL: 0})

duration_corr = work[["duration", "target_yes"]].corr().iloc[0, 1]
print(f"Correlation between duration and target: {duration_corr:.3f}")

display(
    raw_df["pdays"].value_counts().head(10).rename_axis("pdays").reset_index(name="count")
)

pdays_summary = raw_df.assign(was_999=raw_df["pdays"].eq(999)).groupby("was_999")[TARGET_COL].value_counts(normalize=True).mul(100).round(2)
display(pdays_summary.rename("percent").reset_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=raw_df, x="duration", hue=TARGET_COL, bins=40, stat="density", common_norm=False, ax=axes[0])
axes[0].set_title("Duration reveals outcome after the call")
sns.countplot(data=raw_df, x=raw_df["pdays"].eq(999).map({True: "999", False: "contacted"}), hue=TARGET_COL, ax=axes[1])
axes[1].set_title("pdays sentinel split")
axes[1].set_xlabel("pdays group")
plt.tight_layout()
plt.show()


## 9. Numeric Feature Review

The strongest non-leakage numeric signals include previous campaign activity and macroeconomic indicators. These are also useful drift monitoring candidates.


In [ ]:
display(raw_df[numeric_cols].describe().T.round(2))
display(numeric_target_correlation(raw_df).rename("correlation_with_target"))

plot_cols = ["age", "campaign", "pdays", "previous", "emp.var.rate", "euribor3m", "nr.employed"]
raw_df[plot_cols].hist(figsize=(14, 8), bins=30, color="#4C78A8")
plt.suptitle("Selected numeric feature distributions", y=1.02)
plt.tight_layout()
plt.show()


## 10. Categorical Feature Review

These summaries focus on target rate by category. They are more useful than raw counts when deciding whether one-hot encoded categories carry model signal.


In [ ]:
for column in ["job", "education", "contact", "month", "poutcome", "default"]:
    plot_target_rate(raw_df, column, min_count=25, top_n=12)


## 11. Correlation View

This heatmap is intentionally small and numeric-only. It is enough to show the macroeconomic block is correlated, especially `euribor3m`, `emp.var.rate`, and `nr.employed`.


In [ ]:
corr_df = raw_df.copy()
corr_df["target_yes"] = corr_df[TARGET_COL].map({POSITIVE_LABEL: 1, NEGATIVE_LABEL: 0})

plt.figure(figsize=(11, 8))
sns.heatmap(corr_df.select_dtypes(include="number").corr(), cmap="vlag", center=0, annot=False)
plt.title("Numeric correlation matrix")
plt.tight_layout()
plt.show()


## 12. Drift-Relevant Macroeconomic Features

The requirements PDF calls out `euribor3m` and `cons.price.idx` as realistic demo drift levers. The full dataset is ordered by date, so row-order bands give a lightweight view of how these indicators move through the campaign period.


In [ ]:
time_view = raw_df.copy()
time_view["row_band"] = pd.qcut(np.arange(len(time_view)), q=10, labels=[f"band_{i}" for i in range(1, 11)])
time_summary = (
    time_view.groupby("row_band", observed=True)
    .agg(
        euribor3m=("euribor3m", "mean"),
        cons_price_idx=("cons.price.idx", "mean"),
        emp_var_rate=("emp.var.rate", "mean"),
        yes_rate=(TARGET_COL, lambda s: (s == POSITIVE_LABEL).mean() * 100),
    )
    .round(3)
    .reset_index()
)
display(time_summary)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
sns.lineplot(data=time_summary, x="row_band", y="euribor3m", marker="o", ax=axes[0], label="euribor3m")
sns.lineplot(data=time_summary, x="row_band", y="cons_price_idx", marker="o", ax=axes[0], label="cons.price.idx")
axes[0].set_title("Macro indicators over row-order bands")
axes[0].tick_params(axis="x", rotation=45)
sns.lineplot(data=time_summary, x="row_band", y="yes_rate", marker="o", color="#4C78A8", ax=axes[1])
axes[1].set_title("Subscription rate over row-order bands")
axes[1].set_ylabel("Yes rate (%)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 13. EDA Findings

- The target is imbalanced, so threshold tuning is required for a useful recall-oriented operating point.
- `duration` is a clear leakage column and must not be part of serving inputs.
- `pdays == 999` is not a bad value; it is a sentinel for no previous contact and should become an explicit flag.
- `unknown` appears often, especially in `default`, and is kept as a category.
- The macroeconomic indicators are real context variables and are appropriate for drift demos.
- Duplicate rows are rare. We keep them because the requirements do not ask for row deletion and campaign records can naturally repeat feature patterns.


## 14. Cleaning

The cleaning logic now moves from exploratory work into the platform implementation. This keeps the exact same rules available to training, schema generation, drift reference statistics, and service code.


In [ ]:
clean_df = clean_bank_marketing_data(raw_df)

cleaning_checks = pd.DataFrame(
    {
        "check": [
            "duration removed",
            "target encoded as 0/1",
            "pdays_was_999 created",
            "never_contacted_flag created",
            "pdays_clean created",
            "unknown preserved in job",
        ],
        "passed": [
            "duration" not in clean_df.columns,
            set(clean_df[TARGET_COL].unique()).issubset({0, 1}),
            "pdays_was_999" in clean_df.columns,
            "never_contacted_flag" in clean_df.columns,
            "pdays_clean" in clean_df.columns,
            "unknown" in clean_df["job"].unique(),
        ],
    }
)
display(cleaning_checks)
display(clean_df.head())


## 15. Required Split

The requirements PDF specifies a stratified 60/20/20 split with `random_state=42`. The validation split is used for threshold selection. The final test split is used only once for final metrics.


In [ ]:
X, y = split_features_target(clean_df)
X_train, X_val, X_test, y_train, y_val, y_test = make_train_validation_test_split(
    X,
    y,
    train_size=0.60,
    validation_size=0.20,
    test_size=0.20,
    random_state=42,
)

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train), len(X_val), len(X_test)],
        "positive_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
    }
)
split_summary["positive_rate"] = (split_summary["positive_rate"] * 100).round(2)
display(split_summary)


## 16. Training, Tuning, Thresholding, Artifacts, and MLflow

This cell runs the complete platform ML pipeline against the Dockerized MLflow server exposed on `http://localhost:5000`:

- Build preprocessing plus classifier pipelines.
- Cross-validate baseline candidates on the training split.
- Select the baseline family by highest validation threshold satisfying `recall >= 0.75`.
- Tune the selected family.
- Re-apply threshold selection on the validation split.
- Evaluate once on the test split.
- Save `model.pkl`, `schema.json`, `threshold.json`, `metrics.json`, `run_metadata.json`, and `card.md`.
- Log and register the selected model in MLflow.

The platform containers use `http://mlflow:5000` internally, but this notebook runs on the host machine, so it must use the published host port.


In [ ]:
settings = get_ml_settings()
tracking_uri = "http://localhost:5000"

summary = run_training_pipeline(
    data_path=DATA_PATH,
    artifact_root=ARTIFACT_ROOT,
    mlflow_tracking_uri=tracking_uri,
    mlflow_experiment_name=settings.mlflow_experiment_name,
    registered_model_name=settings.mlflow_registered_model_name,
    random_state=42,
    train_size=0.60,
    validation_size=0.20,
    test_size=0.20,
    min_recall=0.75,
    model_version_label=settings.model_version_label,
)

display(summary)


## 17. Artifact Review

After training, these artifacts are the source of truth for serving and audit. The model returns probabilities; `threshold.json` is the operating rule used by the prediction service.


In [ ]:
artifact_dir = ARTIFACT_ROOT / "model_v1"
for path in sorted(artifact_dir.glob("*")):
    print(path.name)

metrics = pd.read_json(artifact_dir / "metrics.json")
threshold = pd.read_json(artifact_dir / "threshold.json", typ="series")

display(metrics)
display(threshold)


## 18. Requirements Traceability

| Requirement | Notebook/platform implementation |
| --- | --- |
| UCI Bank Marketing `bank-additional-full.csv` | `platform/data/bank-additional-full.csv` is the only modeling CSV. |
| Stratified 60/20/20, `random_state=42` | `make_train_validation_test_split` and the notebook split cell. |
| Drop `duration` leakage | `clean_bank_marketing_data` removes it and schema generation excludes it. |
| Treat `pdays == 999` as sentinel | Cleaning creates `pdays_was_999`, `never_contacted_flag`, and `pdays_clean`. |
| Keep `unknown` as category | No missing conversion is applied to `unknown`; one-hot encoding treats it as a valid category. |
| Threshold rule `recall >= 0.75` | Training selects the highest validation threshold meeting recall, then tie-breaks by CV AUC, validation F1, and simplicity. |
| Standard artifact triple | Model binary, schema, model card, metrics, threshold, and metadata are stored under `platform/artifacts/model_v1`. |
| MLflow registration | `run_training_pipeline` logs candidate and final runs and registers the selected model. |
| Drift demo variables | EDA highlights `euribor3m` and `cons.price.idx` as realistic numeric drift levers. |
